In [34]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency, spearmanr
from typing import List, Dict, Tuple


class Full_Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, top_k_missing: int = 80, corr_threshold: float = 0.8):
        self.top_k_missing = top_k_missing
        self.corr_threshold = corr_threshold
        
        self.top_features_ = None
        self.impute_stats_ = None
        self.skew_trans_ = None
        self.cat_manager_ = None
        self.fe_trans_ = None
        self.prune_to_drop_ = None
        self.encoder_ = None
        self.pca_ = None

    # --------------------- 1. LOAD & MEMORY OPTIMIZE ---------------------
    def _load_and_optimize(self, file_path: str) -> pd.DataFrame:
        df = pd.read_csv(file_path)
        # Memory optimize
        float_cols = df.select_dtypes(include='float64').columns
        df[float_cols] = df[float_cols].astype('float32')
        
        # Drop useless columns
        cols_to_drop = ['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_22', 'id_23', 'id_27']
        df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
        return df

    # --------------------- 2. MISSING VALUE HANDLING ---------------------
    def _get_top_missing_features(self, df: pd.DataFrame, target_col="isFraud") -> Tuple[List[str], Dict]:
        fraud_mask = df[target_col] == 1
        missing = df.isnull()
        
        # Vectorized missing rates
        miss_rate = missing.mean()
        miss_fraud = missing[fraud_mask].mean()
        miss_non = missing[~fraud_mask].mean()
        abs_gap_vec = (miss_fraud - miss_non).abs()
        
        rows = []
        for col in df.columns:
            if col == target_col or miss_rate[col] < 0.001:
                continue
                
            # Lấy giá trị đã tính vectorized
            abs_gap = abs_gap_vec[col]
            miss_rate_col = miss_rate[col]
            
            # Fraud rate gap
            is_missing = missing[col]
            fraud_rate_missing = df.loc[is_missing, target_col].mean() if is_missing.any() else 0.0
            fraud_rate_present = df.loc[~is_missing, target_col].mean()
            fraud_rate_gap = abs(fraud_rate_missing - fraud_rate_present)

            # Correlation
            if pd.api.types.is_numeric_dtype(df[col]):
                temp = df[[col, target_col]].copy()
                temp[col] = temp[col].fillna(temp[col].median())
                corr = abs(spearmanr(temp[col], temp[target_col], nan_policy='omit')[0])
            else:
                contingency = pd.crosstab(df[col].fillna("MISSING"), df[target_col], dropna=False)
                if contingency.shape[0] > 1 and contingency.shape[1] > 1:
                    chi2, _, _, _ = chi2_contingency(contingency)
                    n = df.shape[0]
                    min_dim = min(contingency.shape) - 1
                    corr = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0.0
                else:
                    corr = 0.0

            score = (0.35 * abs_gap + 
                    0.30 * fraud_rate_gap + 
                    0.25 * corr + 
                    0.10 * miss_rate_col)
            
            rows.append({"feature": col, "score": score})

        ranking = pd.DataFrame(rows).sort_values("score", ascending=False)
        top_features = ranking.head(self.top_k_missing)["feature"].tolist()

        # Tính impute stats
        impute_stats = {}
        for col in top_features:
            if pd.api.types.is_numeric_dtype(df[col]):
                impute_stats[col] = df[col].median()
            else:
                impute_stats[col] = "MISSING"

        return top_features, impute_stats

    def _preprocess_missing(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        indicator_list = []
        for col in self.top_features_:
            if col in df.columns:
                indicator = df[col].isna().astype("int8")
                indicator.name = f"{col}_isna"
                indicator_list.append(indicator)

        if indicator_list:
            df = pd.concat([df, pd.concat(indicator_list, axis=1)], axis=1)

        for col in self.top_features_:
            if col not in df.columns:
                continue
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(self.impute_stats_.get(col, -999))
            else:
                df[col] = df[col].astype("string").fillna("MISSING").astype("category")
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(-999)
        return df

    # --------------------- 3. SKEW + RARE GROUPING ---------------------
    class SkewedFeatureTransformer(BaseEstimator, TransformerMixin):
        def __init__(self, log_cols=None, clip_percentile=0.99):
            self.log_cols = log_cols or []
            self.clip_percentile = clip_percentile
            self.clip_thresholds_ = {}

        def fit(self, X, y=None):
            num_cols = X.select_dtypes(include=np.number).columns
            for col in num_cols:
                self.clip_thresholds_[col] = X[col].quantile(self.clip_percentile)
            return self

        def transform(self, X):
            X = X.copy()
            for col, th in self.clip_thresholds_.items():
                if col in X.columns:
                    X[col] = X[col].clip(upper=th)
            for col in self.log_cols:
                if col in X.columns:
                    X[col] = np.log1p(X[col])
            return X

    class CategoricalLevelManager(BaseEstimator, TransformerMixin):
        def __init__(self, min_freq=0.0005):
            self.min_freq = min_freq
            self.common_labels_ = {}

        def fit(self, X, y=None):
            cat_cols = X.select_dtypes(exclude=np.number).columns
            for col in cat_cols:
                freq = X[col].value_counts(normalize=True, dropna=True)
                self.common_labels_[col] = freq[freq >= self.min_freq].index.tolist()
            return self

        def transform(self, X):
            X = X.copy()
            for col, common in self.common_labels_.items():
                if col in X.columns:
                    series = X[col].astype(str)
                    is_null = X[col].isna() | (series == 'nan') | (series == 'None')
                    X[col] = np.where(is_null, "MISSING",
                                      np.where(series.isin(common), series, "OTHER"))
            return X

    # --------------------- 4. FEATURE ENGINEERING  ---------------------
    class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
        def __init__(self):
            self.pca = None
            self.card1_stats_ = {}

        def fit(self, X, y=None):
            # card1 amount stats
            self.card1_stats_['mean'] = X.groupby('card1')['TransactionAmt'].mean()
            self.card1_stats_['std'] = X.groupby('card1')['TransactionAmt'].std()
            self.card1_stats_['median'] = X.groupby('card1')['TransactionAmt'].median()

            v_cols = [c for c in X.columns if c.startswith('V')]
            if v_cols:
                self.pca = PCA(n_components=2, random_state=42)
                self.pca.fit(X[v_cols].fillna(-1))
            return self
        @staticmethod
        def _compute_velocity(df: pd.DataFrame):
            df_sorted = df[['card1', 'TransactionDT']].sort_values(['card1', 'TransactionDT']).reset_index(drop=True)
            
            vel1h = np.zeros(len(df_sorted), dtype='int32')
            vel24h = np.zeros(len(df_sorted), dtype='int32')
            
            for card, group in df_sorted.groupby('card1', sort=False):
                idx = group.index.values
                times = group['TransactionDT'].values
                n = len(times)
                left1 = left24 = 0
                
                for i in range(n):
                    while times[left1] < times[i] - 3600 + 1e-6:
                        left1 += 1
                    vel1h[idx[i]] = i - left1 + 1
                    
                    while times[left24] < times[i] - 86400 + 1e-6:
                        left24 += 1
                    vel24h[idx[i]] = i - left24 + 1
            
            original_order = df_sorted.index.argsort()
            return {
                'TransactionVelocity1h': vel1h[original_order],
                'TransactionVelocity24h': vel24h[original_order]
            }

        def transform(self, X):
            df = X.copy()
            new_features = {}
            
            new_features['TransactionAmt_Log'] = np.log1p(df['TransactionAmt'])
            new_features['Amt_decimal'] = df['TransactionAmt'] % 1

            m = df['card1'].map(self.card1_stats_['mean'])
            s = df['card1'].map(self.card1_stats_['std'])
            med = df['card1'].map(self.card1_stats_['median'])

            new_features['card1_Amt_mean'] = m
            new_features['card1_Amt_std'] = s.fillna(0)
            new_features['card1_Amt_median'] = med
            new_features['AmountDeviationUser'] = df['TransactionAmt'] / (m + 0.001)
            std_score = (df['TransactionAmt'] - m) / (s + 0.001)
            new_features['AmountStdScore'] = std_score.fillna(0)
            new_features['AmountToMedianRatio'] = df['TransactionAmt'] / (med + 0.001)

            q95 = df['TransactionAmt'].quantile(0.95)
            new_features['IsLargeTransaction'] = (df['TransactionAmt'] > q95).astype('int8')
            new_features['IsSmallTestTransaction'] = (df['TransactionAmt'] < 5).astype('int8')

            new_features['TransactionHour'] = (df['TransactionDT'] / 3600) % 24
            new_features['TransactionDayOfWeek'] = (df['TransactionDT'] / 86400) % 7
            new_features['IsNightTransaction'] = ((new_features['TransactionHour'] >= 0) & 
                                                (new_features['TransactionHour'] <= 5)).astype('int8')
            new_features['IsWeekendTransaction'] = (new_features['TransactionDayOfWeek'] >= 5).astype('int8')

            new_features['TimeSinceLastTransaction'] = df.groupby('card1')['TransactionDT'].diff().fillna(999999)

            velocity_dict = self._compute_velocity(df)
            new_features.update(velocity_dict)

            card1_agg = df.groupby('card1').agg(
                CardTransactionCount=('TransactionAmt', 'count'),
                D15_mean=('D15', 'mean'),
                dist1_mean=('dist1', 'mean'),
                addr1_nunique=('addr1', 'nunique')
            ).astype('float32')

            new_features['CardTransactionCount'] = df['card1'].map(card1_agg['CardTransactionCount'])
            new_features['D15_to_Mean_card1'] = df.get('D15', 0) / (df['card1'].map(card1_agg['D15_mean']) + 0.001)
            new_features['DistanceDeviation'] = df.get('dist1', 0) - df['card1'].map(card1_agg['dist1_mean'])
            new_features['card1_addr1_unique'] = df['card1'].map(card1_agg['addr1_nunique'])

            new_features['AddressTransactionCount'] = df['addr1'].map(
                df.groupby('addr1')['TransactionAmt'].count()
            )

            new_features['CardAddressCombination'] = df.groupby(['card1', 'addr1'])['TransactionAmt'].transform('count')

            new_features['CardissuerFrequency'] = df['card1'].map(df['card1'].value_counts(dropna=False))
            new_features['DaysSinceRegistration'] = df.get('D1', 0)
            new_features['AccountAgeRisk'] = 1 / (df.get('D1', 1) + 1)
            new_features['TimeSinceLastPurchase'] = df.get('D2', 0)
            new_features['RegistrationToTransactionGap'] = df['TransactionDT'] - df.get('D1', 0)

            new_features['AddrMismatch'] = (df.get('addr1', -1) != df.get('addr2', -2)).astype('int8')
            new_features['IsLongDistance'] = (df.get('dist1', 0) > 100).astype('int8')

            if 'dist1' in df.columns:
                dist_min = df['dist1'].min()
                dist_max = df['dist1'].max()
                new_features['DistanceRiskScore'] = (df['dist1'] - dist_min) / (dist_max - dist_min + 1e-8)
            else:
                new_features['DistanceRiskScore'] = 0.0

            new_features['EmailDomainMatch'] = (df.get('P_emaildomain', '') == df.get('R_emaildomain', '')).astype('int8')
            new_features['IsAnonymousEmail'] = df.get('P_emaildomain', '').isin(['protonmail.com', 'mail.com']).astype('int8')
            new_features['EmailDomainFrequency'] = df.get('P_emaildomain', '').map(
                df['P_emaildomain'].value_counts(dropna=False)
            )
            new_features['IsMobileDevice'] = (df.get('DeviceType', '') == 'mobile').astype('int8')
            new_features['Device_Freq'] = df.get('DeviceInfo', '').map(df['DeviceInfo'].value_counts(dropna=False))

            if 'id_31' in df.columns:
                new_features['id_31_device'] = df['id_31'].str.split(' ', expand=True)[0]

            new_features['CardIPCount'] = df.get('C5', 0)
            new_features['AddressDeviceCount'] = df.get('C7', 0)
            new_features['AssociationRatio'] = (df.get('C1', 0) + df.get('C2', 0)) / (df.get('C3', 0) + 0.01)
            new_features['TotalAssociations'] = df.get('C1', 0) + df.get('C2', 0) + df.get('C3', 0)
            new_features['C1_count'] = df.get('C1', 0).map(df['C1'].value_counts(dropna=False))

            if self.pca is not None:
                v_cols = [c for c in df.columns if c.startswith('V')]
                if v_cols:
                    v_pca = self.pca.transform(df[v_cols].fillna(-1))
                    new_features['V_PCA_1'] = v_pca[:, 0]
                    new_features['V_PCA_2'] = v_pca[:, 1]

            # final concat
            new_df = pd.DataFrame(new_features, index=df.index)
            df = pd.concat([df, new_df], axis=1)
            df.replace([np.inf, -np.inf], np.nan, inplace=True)
            
            return df

    # --------------------- 5. PRUNING & ENCODING ---------------------
    def _prune_correlated(self, df: pd.DataFrame, target_col="isFraud"):
        numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if c != target_col]
        if len(numeric_cols) < 2:
            return df, []
        numeric_data = df[numeric_cols].values.astype('float32')
        corr_matrix = np.corrcoef(numeric_data, rowvar=False)
        upper = np.triu(corr_matrix, k=1)
        upper_df = pd.DataFrame(upper, index=numeric_cols, columns=numeric_cols)
        high_corr = upper_df.stack().reset_index()
        high_corr.columns = ["f1", "f2", "corr"]
        high_corr = high_corr[high_corr["corr"] >= self.corr_threshold]

        target_corr = {col: abs(df[col].corr(df[target_col], method="spearman")) for col in numeric_cols}
        protected_cols = ['TransactionAmt', 'TransactionDT', 'card1', 'addr1', 'ProductCD', 
                            'D1', 'D15']
        to_drop = set()
        for _, row in high_corr.sort_values("corr", ascending=False).iterrows():
            f1, f2 = row["f1"], row["f2"]
            if f1 in protected_cols or f2 in protected_cols:
                if f1 in protected_cols:
                    to_drop.add(f2)
                else:
                    to_drop.add(f1)
                continue
            if target_corr.get(f1, 0) >= target_corr.get(f2, 0):
                to_drop.add(f2)
            else:
                to_drop.add(f1)
        return df.drop(columns=list(to_drop)), list(to_drop)

    class CategoricalEncoder(BaseEstimator, TransformerMixin):
        def __init__(self):
            self.encoders_ = {}

        def fit(self, X, y=None):
            cat_cols = X.select_dtypes(exclude=np.number).columns
            for col in cat_cols:
                le = LabelEncoder()
                le.fit(X[col].astype(str))
                self.encoders_[col] = le
            return self

        def transform(self, X):
            X = X.copy()
            for col, le in self.encoders_.items():
                if col in X.columns:
                    X[col] = X[col].astype(str).map(lambda s: s if s in le.classes_ else 'RARE')
                    X[col] = le.transform(X[col])
            return X

    # ====================== MAIN FIT / TRANSFORM ======================
    def fit(self, X: pd.DataFrame, y=None):
        if isinstance(X, str):
            df = self._load_and_optimize(X)
        else:
            df = X.copy() if isinstance(X, pd.DataFrame) else X
        df['null_counts'] = df.isnull().sum(axis=1).astype('int16')
        self.top_features_, self.impute_stats_ = self._get_top_missing_features(df)
        df = self._preprocess_missing(df)

        self.skew_trans_ = self.SkewedFeatureTransformer(log_cols=['TransactionAmt', 'C1', 'C2'])
        df = self.skew_trans_.fit_transform(df)

        self.cat_manager_ = self.CategoricalLevelManager()
        df = self.cat_manager_.fit_transform(df)

        self.fe_trans_ = self.FeatureEngineeringTransformer()
        df = self.fe_trans_.fit_transform(df)

        df, self.prune_to_drop_ = self._prune_correlated(df)
        self.encoder_ = self.CategoricalEncoder()
        self.encoder_.fit(df)

        print(f"Pipeline fitted successfully. Final shape: {df.shape}")
        return self

    def transform(self, X: pd.DataFrame):
        if isinstance(X, str):
            df = self._load_and_optimize(X)          
        else:
            df = X.copy() if isinstance(X, pd.DataFrame) else X
        df['null_counts'] = df.isnull().sum(axis=1).astype('int16')
        df = self._preprocess_missing(df)
        df = self.skew_trans_.transform(df)
        df = self.cat_manager_.transform(df)
        df = self.fe_trans_.transform(df)
        df = df.drop(columns=self.prune_to_drop_, errors='ignore')
        df = self.encoder_.transform(df)
        return df

# ====================== USAGE ======================
if __name__ == "__main__":
    preprocessor = Full_Preprocessor(top_k_missing=80, corr_threshold=0.95)
    df_final = preprocessor.fit_transform("../data/merged_train_data.csv") 

C:\Users\HP\AppData\Local\Temp\ipykernel_21468\1305772399.py:366: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['null_counts'] = df.isnull().sum(axis=1).astype('int16')
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1661: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Pipeline fitted successfully. Final shape: (590540, 195)


C:\Users\HP\AppData\Local\Temp\ipykernel_21468\1305772399.py:391: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['null_counts'] = df.isnull().sum(axis=1).astype('int16')


In [36]:
print(df_final.shape)
print(df_final.info())
print("Target distribution:\n", df_final['isFraud'].value_counts(normalize=True))

(590540, 195)
<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 195 entries, isFraud to V_PCA_2
dtypes: float32(122), float64(18), int16(1), int32(2), int64(35), int8(17)
memory usage: 528.8 MB
None
Target distribution:
 isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [39]:
df_final.head()

,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,dist1,dist2,P_emaildomain,R_emaildomain,C2,C3,C4,C5,C7,C8,C10,C11,C12,C13,D1,D2,D3,D5,D6,D7,D8,D10,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V10,V29,V52,V69,V90,V103,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V140,V150,V159,V165,V167,V168,V170,V171,V176,V177,V178,V180,V184,V185,V186,V187,V189,V190,V191,V192,V193,V199,V201,V204,V211,V212,V215,V217,V218,V219,V222,V228,V229,V230,V232,V234,V239,V242,V243,V244,V246,V247,V248,V249,V257,V258,V261,V263,V265,V274,V283,V294,V303,V306,V307,V308,V309,V310,V311,V312,V314,V315,V316,V317,V318,V319,V320,V321,id_01,id_02,id_03,id_06,id_10,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_28,id_29,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,null_counts,V244_isna,R_emaildomain_isna,id_35_isna,D12_isna,DeviceInfo_isna,D14_isna,id_16_isna,D8_isna,D6_isna,Amt_decimal,card1_Amt_std,card1_Amt_median,AmountStdScore,IsLargeTransaction,IsSmallTestTransaction,TransactionHour,TransactionDayOfWeek,IsNightTransaction,IsWeekendTransaction,TimeSinceLastTransaction,TransactionVelocity1h,TransactionVelocity24h,CardTransactionCount,D15_to_Mean_card1,DistanceDeviation,card1_addr1_unique,AddressTransactionCount,CardAddressCombination,AccountAgeRisk,IsLongDistance,EmailDomainMatch,IsAnonymousEmail,EmailDomainFrequency,IsMobileDevice,id_31_device,C1_count,V_PCA_1,V_PCA_2
0,0,86400.0,4.241327,4,13926.0,-999.0,150.0,2,142.0,2,315.0,19.0,-999.0,0,0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1.0,14.0,-999.0,13.0,-999.0,0.0,-999.0,37.875,13.0,0.0,-999.0,0.0,0.0,2,2,2,2,0,2,1,1,1,0.0,0.0,-999.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,-999.0,-999.0,-999.000000,-999.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-999.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-999.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,-5.0,-999.0,-999.0,0.0,-999.0,1,-999.0,-999.0,1,1,166.0,-999.0,1,1,12,0,55,0,1,1,1,1,0,1,225,1,1,1,1,1,1,1,1,1,0.241327,1.032028,5.017280,-1.056778,0,1,0.000000,1.000000,1,0,999999.0,1,1,43.0,-0.000000,680.534912,11.0,23078,3,0.066667,0,1,0,94456,0,0,316791,-5144.782762,-1446.635797
1,0,86401.0,3.401197,4,2755.0,404.0,150.0,3,102.0,2,325.0,-999.0,-999.0,14,0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,-999.0,-999.0,-999.0,0.0,-999.0,37.875,0.0,0.0,-999.0,0.0,0.0,0,1,1,0,2,2,1,1,1,-999.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-999.0,-999.0,-999.000000,-999.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-999.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-999.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-5.0,-999.0,-999.0,0.0,-999.0,1,-999.0,-999.0,1,1,166.0,-999.0,1,1,12,0,55,0,1,1,1,1,0,1,221,1,1,1,1,1,1,1,1,1,0.401197,0.912662,4.736198,-1.555986,0,1,0.000278,1.000012,1,0,999999.0,1,1,683.0,0.000000,-544.722473,44.0,42751,61,1.000000,0,0,0,228355,0,0,316791,-5164.224258,-1792.120890
2,0,86469.0,4.094345,4,4663.0,490.0,150.0,4,166.0,3,330.0,287.0,-999.0,27,0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,-999.0,-999.0,-999.0,0.0,-999.0,37.875,0.0,0.0,-999.0,0.0,315.0,2,2,2,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-999.0,-999.0,-999.000000,-999.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-999.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-999.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-5.0,-999.0,-999.0,0.0,-999.0,1,-999.0,-999.0,1,1,166.0,-999.0,1,1,12,0,55,0,1,1,1,1,0,1,202,1,1,1,1,1,1,1,1,1,0.094345,0.748148,4.094345,-0.227625,0,1,0.019167,1.000799,1,0,999999.0,1,1,1108.0,4.615317,770.562775,44.0,26287,34,1.000000,1,0,0,5096,0,0,316791,-5183

In [44]:
import joblib
from pathlib import Path

model_path = Path("../models")
model_path.mkdir(exist_ok=True)
data_path = Path("../data")

joblib.dump(preprocessor, model_path / "preprocessor_v1.pkl")        
df_final.to_parquet(data_path / "preprocessed_train.parquet", index=False)

In [45]:
df_final = df_final.sort_values('TransactionDT').reset_index(drop=True)

split_idx = int(len(df_final) * 0.8)
train_df = df_final.iloc[:split_idx]
val_df   = df_final.iloc[split_idx:]

print(f"Train shape: {train_df.shape} | Val shape: {val_df.shape}")
print(f"Train fraud rate: {train_df['isFraud'].mean():.4f}")
print(f"Val fraud rate  : {val_df['isFraud'].mean():.4f}")


target = 'isFraud'
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val   = val_df.drop(columns=[target])
y_val   = val_df[target]

X_train.to_parquet(data_path / "X_train.parquet", index=False)
y_train.to_frame().to_parquet(data_path / "y_train.parquet", index=False)
X_val.to_parquet(data_path / "X_val.parquet", index=False)
y_val.to_frame().to_parquet(data_path / "y_val.parquet", index=False)

Train shape: (472432, 195) | Val shape: (118108, 195)
Train fraud rate: 0.0351
Val fraud rate  : 0.0344
